# Homework 6: Decision Trees and Ensemble Learning

### Import Library & Data Preparation
Load `car_fuel_efficiency_2026.csv` and apply the reference missing-value imputation policy:
- Fill missing values with zeros.
- Do the split with these exact calls:
```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
```
- Use `DictVectorizer(sparse=True)` to turn dataframes into matrices.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

# Load dataset and fill missing values with zeros
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df = df.fillna(0)

# Exact train/validation/test split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

target = 'fuel_efficiency_mpg'

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

del df_train[target]
del df_val[target]
del df_test[target]

train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')
test_dicts = df_test.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

Train shape: (6000, 16), Val shape: (2000, 16), Test shape: (2000, 16)

---
### Q1. Original Splitting Feature

Let's train a decision tree regressor to predict the `fuel_efficiency_mpg` variable.
- Train a model with `max_depth=1`.

Which original feature is used for splitting the data? If you use `DictVectorizer`, map a one-hot feature such as `origin=USA` back to its original column name.
- `'vehicle_weight'`
- `'model_year'`
- `'origin'`
- `'fuel_type'`

In [2]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

feature_names = list(dv.get_feature_names_out())
tree_text = export_text(dt, feature_names=feature_names)
print(tree_text)

# Identify the root split feature
root_feature = tree_text.strip().split()[1]
print(f"Identified split feature: {root_feature}")
print(f"Answer: '{root_feature}'")

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]

Identified split feature: model_year
Answer: 'model_year'

**Answer: `'model_year'`**  
The decision tree with `max_depth=1` splits at the root node using the condition `model_year <= 1997.50`.

---
### Q2. Training the Random Forest Regressor

Train a random forest regressor with these parameters:
- `n_estimators=10`
- `random_state=1`
- `n_jobs=-1` (optional - to make training faster)

What's the RMSE of this model on the validation data?
- 0.1837
- 1.837
- 18.37
- 183.7

In [3]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse_q2 = root_mean_squared_error(y_val, y_pred)
print(f"Validation RMSE (n_estimators=10): {rmse_q2:.4f}")

# Match with options:
options = [0.1837, 1.837, 18.37, 183.7]
closest_opt = min(options, key=lambda x: abs(x - rmse_q2))
print(f"Closest option: {closest_opt}")
print(f"Answer: {closest_opt} (exact: {rmse_q2:.4f})")

Validation RMSE (n_estimators=10): 1.8394
Closest option: 1.837
Answer: 1.837 (exact: 1.8394)

**Answer: `1.837`**  
The validation RMSE of the Random Forest Regressor with 10 trees is approximately `1.8394`, which matches the option `1.837`.

---
### Q3. Experimenting with n_estimators

Now let's experiment with the `n_estimators` parameter:
- Try these values of this parameter: `[10, 50, 100, 150]`.
- Set `random_state` to `1`.
- Evaluate the model on the validation dataset.

Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.
- 10
- 50
- 100
- 150

In [4]:
scores_n_est = {}

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    scores_n_est[n] = rmse
    print(f"n_estimators={n:3d} -> Validation RMSE: {rmse:.4f} (exact: {rmse:.6f})")

best_n = min(scores_n_est, key=scores_n_est.get)
print(f"\nLowest validation RMSE: {scores_n_est[best_n]:.4f} achieved with n_estimators={best_n}")
print(f"Answer: {best_n}")

n_estimators= 10 -> Validation RMSE: 1.8394 (exact: 1.839383)
n_estimators= 50 -> Validation RMSE: 1.7729 (exact: 1.772863)
n_estimators=100 -> Validation RMSE: 1.7676 (exact: 1.767643)
n_estimators=150 -> Validation RMSE: 1.7681 (exact: 1.768109)

Lowest validation RMSE: 1.7676 achieved with n_estimators=100
Answer: 100

**Answer: `100`**  
Validation RMSE drops significantly from `1.8394` at `n_estimators=10` to `1.7676` at `n_estimators=100`, giving the lowest validation error among all evaluated choices.

---
### Q4. Selecting the Best max_depth

Let's select the best `max_depth`:
- Try different values of `max_depth`: `[10, 15, 20, 25]`
- For each of these values,
  - try different values of `n_estimators` from `[10, 50, 100, 150]`
  - calculate the mean RMSE 
- Fix the random seed: `random_state=1`

What's the best `max_depth`, using the mean RMSE?
- 10
- 15
- 20
- 25

In [5]:
depth_scores = {}

for d in [10, 15, 20, 25]:
    rmses = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmses.append(root_mean_squared_error(y_val, y_pred))
    mean_rmse = np.mean(rmses)
    depth_scores[d] = mean_rmse
    print(f"max_depth={d:2d} -> Mean RMSE: {mean_rmse:.4f} (scores: {[round(r, 4) for r in rmses]})")

best_depth = min(depth_scores, key=depth_scores.get)
print(f"\nBest max_depth by mean RMSE: {best_depth} (mean RMSE = {depth_scores[best_depth]:.4f})")
print(f"Answer: {best_depth}")

max_depth=10 -> Mean RMSE: 1.7541 (scores: [1.7821, 1.7471, 1.7436, 1.7438])
max_depth=15 -> Mean RMSE: 1.7837 (scores: [1.8297, 1.7712, 1.7672, 1.7666])
max_depth=20 -> Mean RMSE: 1.7878 (scores: [1.84, 1.7744, 1.7692, 1.7678])
max_depth=25 -> Mean RMSE: 1.7867 (scores: [1.84, 1.7727, 1.7667, 1.7673])

Best max_depth by mean RMSE: 10 (mean RMSE = 1.7541)
Answer: 10

**Answer: `10`**  
`max_depth=10` achieves the lowest mean validation RMSE (`1.7541`). Shallower trees provide better regularization against overfitting the training set compared to deeper trees (depths 15, 20, 25).

---
### Q5. Feature Importance

We can extract feature importance information from tree-based models via the `feature_importances_` field.
- Train the model with these parameters:
  - `n_estimators=10`,
  - `max_depth=20`,
  - `random_state=1`,
  - `n_jobs=-1` (optional)
- Get the feature importance information from this model

What's the most important feature (among these 4)?
- `vehicle_weight`
- `horsepower`
- `acceleration`
- `engine_displacement`

In [6]:
rf_q5 = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_q5.fit(X_train, y_train)

feature_importances = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf_q5.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print("Top 10 most important features:")
print(feature_importances.head(10))

candidate_features = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
df_candidates = feature_importances[feature_importances['feature'].isin(candidate_features)].sort_values('importance', ascending=False)
print("\nComparison among the 4 candidate features:")
print(df_candidates)

top_candidate = df_candidates.iloc[0]['feature']
top_importance = df_candidates.iloc[0]['importance']
print(f"\nMost important feature among candidates: {top_candidate} ({top_importance:.4f})")
print(f"Answer: {top_candidate}")

Top 10 most important features:
                        feature  importance
0                    model_year    0.295915
1                vehicle_weight    0.203067
2            fuel_type=Gasoline    0.160712
3                    horsepower    0.078468
4           engine_displacement    0.061152
5                  acceleration    0.055342
6                    origin=USA    0.042268
7  drivetrain=Front-wheel drive    0.020108
8                     num_doors    0.019364
9              fuel_type=Diesel    0.018045

Comparison among the 4 candidate features:
               feature  importance
1       vehicle_weight    0.203067
3           horsepower    0.078468
4  engine_displacement    0.061152
5         acceleration    0.055342

Most important feature among candidates: vehicle_weight (0.2031)
Answer: vehicle_weight

**Answer: `vehicle_weight`**  
Among the four candidate features, `vehicle_weight` has the highest importance score (`0.2031`), followed by `horsepower` (`0.0785`), `engine_displacement` (`0.0612`), and `acceleration` (`0.0553`).

---
### Q6. Tuning XGBoost eta

Now let's train an XGBoost model! For this question, we'll tune the `eta` parameter:
- Install XGBoost
- Create DMatrix for train and validation
- Create a watchlist
- Train a model with these parameters for 100 rounds:
```python
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}
```
Now change `eta` from `0.3` to `0.1`.

Which eta leads to the best RMSE score on the validation dataset?
- 0.3
- 0.1
- Both give equal value

In [7]:
clean_features = [f.replace('[', '_').replace(']', '_').replace('<', '_') for f in dv.get_feature_names_out()]

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=clean_features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=clean_features)

watchlist = [(dtrain, 'train'), (dval, 'val')]
val_rmse_results = {}

for eta in [0.3, 0.1]:
    evals_result = {}
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        evals_result=evals_result,
        verbose_eval=False
    )
    
    y_pred = model.predict(dval)
    val_rmse = root_mean_squared_error(y_val, y_pred)
    val_rmse_results[eta] = val_rmse
    print(f"eta={eta} -> Validation RMSE at round 100: {val_rmse:.4f} (exact: {val_rmse:.6f})")

best_eta = min(val_rmse_results, key=val_rmse_results.get)
print(f"\nBest eta: {best_eta} (RMSE = {val_rmse_results[best_eta]:.4f})")
print(f"Answer: {best_eta}")

eta=0.3 -> Validation RMSE at round 100: 1.8265 (exact: 1.826458)
eta=0.1 -> Validation RMSE at round 100: 1.7248 (exact: 1.724810)

Best eta: 0.1 (RMSE = 1.7248)
Answer: 0.1

**Answer: `0.1`**  
At round 100, `eta=0.1` yields a validation RMSE of `1.7248`, outperforming `eta=0.3` which overfits and reaches `1.8265`. Therefore, `eta=0.1` leads to the best RMSE score on the validation dataset.